## Phase 4: Habitability Scoring

Combines two planet sets into one unified table:
1. **CONFIRMED Planets** from the PSCompPars table (Kepler + K2 + TESS + ground-based)
2. **CANDIDATE rows predicted as likely real** from Phase 3 (`predicted_real == 1`)

Computes, for each planet:
- Habitable zone status (conservative and optimistic bounds, based on insolation flux)
- A simplified Earth Similarity Index (ESI) Score

**Important scope note**: this notebook estimates whether a planet sits in a zone where liquid water *could* exist on a rocky world, based on standard astronomical habitable-zone criteria. It does not and cannot assess breathable atmosphere, gravity, radiation, or actual human survivability - "habitable zone" is a well-defined astronomical term, not a claim about livability for humans

In [2]:
import pandas as pd
import numpy as np

confirmed = pd.read_csv("../data/raw/confirmed_planets.csv", comment="#", low_memory=False)
candidates_pred = pd.read_csv("../data/processed/koi_candidates_predicted.csv")

likely_real_candidates = candidates_pred[candidates_pred["predicted_real"] == 1].copy()

print("Confirmed planets:", confirmed.shape)
print("Likely-real candidates:", likely_real_candidates.shape)

Confirmed planets: (6366, 703)
Likely-real candidates: (1267, 113)


### Harmonize column Names

The two source tables use ddifferent naming conventions (`pl_*`/`st_*` for PSCompPars vs `koi_*` for KOI). Building a common schema with the columns needed for habitability scoring.

In [3]:
confirmed_habitability = pd.DataFrame({
    "planet_name": confirmed["pl_name"],
    "source": "confirmed",
    "insolation_flux": confirmed["pl_insol"],
    "eq_temp_k": confirmed["pl_eqt"],
    "radius_earth": confirmed["pl_rade"],
    "mass_earth": confirmed.get("pl_bmasse", np.nan),
    "orbital_period_days": confirmed["pl_orbper"],
    "st_teff_k": confirmed["st_teff"],
    "discovery_facility": confirmed.get("disc_facility", np.nan)
})

candidate_habitability = pd.DataFrame({
    "planet_name": "KOI-" + likely_real_candidates["kepid"].astype(str),
    "source": "candidates_predicted_real",
    "insolation_flux": likely_real_candidates["koi_insol"],
    "eq_temp_k": likely_real_candidates["koi_teq"],
    "radius_earth": likely_real_candidates["koi_prad"],
    "mass_earth": np.nan,
    "orbital_period_days": likely_real_candidates["koi_period"],
    "st_teff_k": likely_real_candidates["koi_steff"],
    "discovery_facility": "Kepler (candidate)"
})

combined = pd.concat([confirmed_habitability, candidate_habitability], ignore_index=True)
print("Combined shape:", combined.shape)
print("\nSource Breakdown:")
print(combined["source"].value_counts())

Combined shape: (7633, 9)

Source Breakdown:
source
confirmed                    6366
candidates_predicted_real    1267
Name: count, dtype: int64


### Drop Rows Missing Core Habitability Inputs

Insolation flux and equilibrium temperature are required to assess habitable-zone status - rows missing both cannot be evaluated.

In [4]:
before = len(combined)
combined = combined.dropna(subset=["insolation_flux", "radius_earth"]).copy()
after = len(combined)

print(f"Combined {before - after} rows missing insolation flux or radius")
print(f"Remaining: {after} rows")

Combined 629 rows missing insolation flux or radius
Remaining: 7004 rows


### Habitable Zone Classification

Using insolation-flux-based habitable zone bounds (relative to Earth = 1.0 S⊕), following the commonly used Kopparapu et al. framework, simplified to fixed bounds:

- **Conservative HZ**: 0.36 - 1.11 S⊕ (runaway greenhouse to maximum greenhouse limits)
- **Optimistic HZ**: 0.25 - 1.5 S⊕ (recent Venus to early Mars limits)

This is a standard simplication - precise bounds shift slightly with host star temperature, which is not modeled here.

In [5]:
def classify_hz(insol):
    if pd.isna(insol):
        return "unknown"
    if 0.36 <= insol <= 1.11:
        return "conservative_hz"
    elif 0.25 <= insol <= 1.5:
        return "optimistic_hz"
    else:
        return "outside_hz"

combined["hz_status"] = combined["insolation_flux"].apply(classify_hz)
print(combined["hz_status"].value_counts())
print(f"\nPercent in conservative HZ: {(combined['hz_status'] == 'conservative_hz').mean() * 100:.2f}%")
print(f"Percent in conservative or optimistic HZ: {combined['hz_status'].isin(['conservative_hz', 'optimistic_hz']).mean() * 100:.2f}%")

hz_status
outside_hz         6664
conservative_hz     225
optimistic_hz       115
Name: count, dtype: int64

Percent in conservative HZ: 3.21%
Percent in conservative or optimistic HZ: 4.85%


### Radius-Based Rocky Planet Filter

Habitable zone status alone isn't enough - a gas giant in a habitable zone isn't Earth-like. Filtering to plausibly rocky planets (radius < 1.6 Earth radii, the commonly used rocky/non-rocky transistion threshold from Kepler-era studies)

In [6]:
combined["likely_rocky"] = combined["radius_earth"] < 1.6

rocky_hz = combined[(combined["likely_rocky"]) & (combined["hz_status"].isin(["conservative_hz", "optimistic_hz"]))]
print(f"Rocky + in HZ (conservative or optimistic): {len(rocky_hz)} planets out of {len(combined)} total ({len(rocky_hz)/len(combined)*100:.2f}%)")
rocky_hz[["planet_name", "source", "insolation_flux", "eq_temp_k", "radius_earth", "hz_status"]].sort_values("insolation_flux")

Rocky + in HZ (conservative or optimistic): 38 planets out of 7004 total (0.54%)


,planet_name,source,insolation_flux,eq_temp_k,radius_earth,hz_status
348,TRAPPIST-1 g,confirmed,0.2520,197.30,1.129,optimistic_hz
5681,GJ 1002 c,confirmed,0.2570,181.70,1.100,optimistic_hz
7597,KOI-9210820,candidates_predicted_real,0.2900,187.00,1.370,optimistic_hz
2751,Kepler-186 f,confirmed,0.3000,177.00,1.170,optimistic_hz
2285,GJ 667 C e,confirmed,0.3022,NaN,1.450,optimistic_hz
905,Teegarden's Star c,confirmed,0.3500,209.00,1.020,optimistic_hz
7583,KOI-10452252,candidates_predicted_real,0.3700,199.00,0.420,conservative_hz
347,TRAPPIST-1 f,confirmed,0.3730,217.70,1.045,conservative_hz
7589,KOI-9084569,candidates_predicted_real,0.4400,208.00,0.970,conservative_hz
145,Kepler-62 f,confirmed,0.5000,208.00,1.410,conservative_hz


### Simplified Earth Similarity Index (ESI)

ESI compares a planet to Earth across physical dimensions, scored 0(no similarity) to 1 (identified to Earth). This is a simplified version using radius and equilibrium temperature only (the two most reliably available parameters here), following the standard ESI weighting formula.

In [8]:
EARTH_RADIUS = 1.0
EARTH_TEMP = 288.0

def esi_component(value, earth_value, weight):
    if pd.isna(value) or value <= 0:
        return np.nan
    return (1 - abs((value - earth_value) / (value + earth_value))) ** weight

combined["esi_radius"] = combined["radius_earth"].apply(lambda x: esi_component(x, EARTH_RADIUS, 0.57))
combined["esi_temp"] = combined["eq_temp_k"].apply(lambda x: esi_component(x, EARTH_TEMP, 5.58))

combined["esi_score"] = (combined["esi_radius"] * combined["esi_temp"]) ** 0.5

combined["esi_score"].describe()

count    6998.000000
mean        0.208992
std         0.189973
min         0.000002
25%         0.067317
50%         0.145054
75%         0.301045
max         0.983530
Name: esi_score, dtype: float64

### Top Candidates by ESI Score

In [9]:
top_esi = combined.dropna(subset=["esi_score"]).sort_values("esi_score", ascending=False).head(20)
top_esi[["planet_name", "source", "radius_earth", "eq_temp_k", "insolation_flux", "hz_status", "esi_score"]]

,planet_name,source,radius_earth,eq_temp_k,insolation_flux,hz_status,esi_score
6150,Kepler-438 b,confirmed,1.120,288.00,1.400,optimistic_hz,0.983530
1653,GJ 3323 b,confirmed,1.230,290.06,2.580,outside_hz,0.959845
332,TRAPPIST-1 d,confirmed,0.788,286.20,1.115,optimistic_hz,0.956257
3551,K2-72 e,confirmed,1.290,286.34,1.200,optimistic_hz,0.954406
2420,Kepler-138 e,confirmed,0.797,292.00,1.730,outside_hz,0.947933
5875,Teegarden's Star b,confirmed,1.050,277.00,1.080,conservative_hz,0.939985
7372,KOI-8364232,candidates_predicted_real,1.310,295.00,1.790,outside_hz,0.927956
548,L 98-59 f,confirmed,1.480,285.00,1.100,conservative_hz,0.926860
3570,Ross 128 b,confirmed,1.110,301.00,1.380,optimistic_hz,0.925401
475,TOI-700 e,confirmed,0.953,272.90,1.270,optimistic_hz,0.920276


### Summary Statistics for the Report

Key numbers: what fraction of all evaluated planets fall in the habitable zone, and how that breaks down by source (confirmed vs. predicted-real candidates).

In [10]:
summary = combined.groupby("source").agg(
    total=("planet_name", "count"),
    in_conservative_hz = ("hz_status", lambda x: (x == "conservative_hz").sum()),
    in_optimistic_hz = ("hz_status", lambda x: (x == "optimistic_hz").sum()),
    rocky_count = ("likely_rocky", "sum")
)

summary["pct_conservative_hz"] = (summary["in_conservative_hz"] / summary["total"] * 100).round(2)
summary

,total,in_conservative_hz,in_optimistic_hz,rocky_count,pct_conservative_hz
source,,,,,
candidates_predicted_real,1267,40,13,653,3.16
confirmed,5737,185,102,1103,3.22


### Saved Processed Habitability Dataset

In [11]:
combined.to_csv("../data/processed/habitatbility_scores.csv", index=False)
rocky_hz.to_csv("../data/processed/rocky_habitable_zone_planets.csv", index=False)

print("Saved habitability_scores.csv", combined.shape)
print("Saved rocky_habitable_zone_planets.csv", rocky_hz.shape)

Saved habitability_scores.csv (7004, 14)
Saved rocky_habitable_zone_planets.csv (38, 11)
